# CNV / ploidy post-processing -- GC.PS.1929.WGS

Parental / Resistant / Fused NSCLC cell line WGS, tumor-only, purity fixed at 1.0.

Inputs (produced by `scripts/run_cnv_ploidy_pipeline.sh`):
- `analysis/cnv_ploidy/cnvkit/{name}.cnr` -- bin-level log2 ratios
- `analysis/cnv_ploidy/cnvkit/{name}.cns` -- segments
- `analysis/cnv_ploidy/cnvkit/{name}.call.cns` -- integer copy number calls
- `analysis/cnv_ploidy/baf/{name}.vcf.gz` -- pseudo-BAF at common SNP sites
- `analysis/cnv_ploidy/ichorcna/{name}/` -- independent purity/ploidy estimate

This notebook: (1) plots genome-wide profiles per sample, (2) derives an
empirical ploidy estimate from the segment log2 distribution, (3) computes
relative CN of Resistant/Fused vs. Parental to flag acquired events, and
(4) overlays pseudo-BAF for LOH context.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

INPUT_DIR = Path("analysis/cnv_ploidy")  # adjust if running elsewhere
CNVKIT_DIR = INPUT_DIR / "cnvkit"
BAF_DIR = INPUT_DIR / "baf"

SAMPLES = ["parental", "resistant", "fused"]
COLORS = {"parental": "#4C72B0", "resistant": "#C44E52", "fused": "#55A868"}

CHROM_ORDER = [str(i) for i in range(1, 23)] + ["X"]

In [ ]:
def load_cnr(name):
    df = pd.read_csv(CNVKIT_DIR / f"{name}.cnr", sep="\t")
    df["chromosome"] = df["chromosome"].astype(str).str.replace("chr", "", regex=False)
    df = df[df["chromosome"].isin(CHROM_ORDER)].copy()
    df["chromosome"] = pd.Categorical(df["chromosome"], categories=CHROM_ORDER, ordered=True)
    return df.sort_values(["chromosome", "start"]).reset_index(drop=True)

def load_cns(name):
    df = pd.read_csv(CNVKIT_DIR / f"{name}.cns", sep="\t")
    df["chromosome"] = df["chromosome"].astype(str).str.replace("chr", "", regex=False)
    df = df[df["chromosome"].isin(CHROM_ORDER)].copy()
    return df

def add_genome_coord(df, chrom_sizes):
    offsets = {}
    running = 0
    for c in CHROM_ORDER:
        offsets[c] = running
        running += chrom_sizes.get(c, 0)
    df = df.copy()
    df["genome_start"] = df.apply(lambda r: r["start"] + offsets[str(r["chromosome"])], axis=1)
    return df, offsets

cnr = {s: load_cnr(s) for s in SAMPLES}
cns = {s: load_cns(s) for s in SAMPLES}

# approximate chrom sizes from max bin end observed across samples (good enough for plotting order)
chrom_sizes = (
    pd.concat([cnr[s][["chromosome", "end"]] for s in SAMPLES])
    .groupby("chromosome")["end"].max().to_dict()
)
for s in SAMPLES:
    cnr[s], offsets = add_genome_coord(cnr[s], chrom_sizes)
    cns[s], _ = add_genome_coord(cns[s], chrom_sizes)

## 1. Genome-wide log2 profiles (all three samples stacked)

In [ ]:
fig, axes = plt.subplots(len(SAMPLES), 1, figsize=(16, 3 * len(SAMPLES)), sharex=True)

chrom_boundaries = [offsets[c] for c in CHROM_ORDER] + [max(offsets.values()) + chrom_sizes[CHROM_ORDER[-1]]]

for ax, s in zip(axes, SAMPLES):
    ax.scatter(cnr[s]["genome_start"], cnr[s]["log2"], s=1, alpha=0.3, color=COLORS[s])
    ax.hlines(cns[s]["log2"], cns[s]["genome_start"], cns[s]["genome_start"] + (cns[s]["end"] - cns[s]["start"]),
              color="black", linewidth=2)
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.set_ylim(-2, 2)
    ax.set_ylabel(f"{s}\nlog2 ratio")
    for b in chrom_boundaries:
        ax.axvline(b, color="grey", linewidth=0.3)

axes[-1].set_xticks([offsets[c] for c in CHROM_ORDER])
axes[-1].set_xticklabels(CHROM_ORDER, fontsize=8)
axes[-1].set_xlabel("Chromosome")
plt.tight_layout()
plt.savefig(INPUT_DIR / "genome_wide_log2_all_samples.png", dpi=200)
plt.show()

## 2. Empirical ploidy estimate from the segment log2 distribution

Length-weighted histogram of segment log2 values, clustered with a Gaussian
mixture to find the dominant copy-number states. The tallest peak is the
modal (most common) copy number in the genome; average ploidy is the
length-weighted mean copy number across all segments once that peak is
anchored to an integer state.

This is a computational estimate only -- treat it as a starting point and
confirm against flow cytometry DNA content if you run that assay.

In [ ]:
from sklearn.mixture import GaussianMixture

def estimate_ploidy(cns_df, assumed_neutral_cn=2, n_states=6):
    seg = cns_df.copy()
    seg["length"] = seg["end"] - seg["start"]
    seg = seg[seg["length"] > 0]

    weights = seg["length"].values
    log2 = seg["log2"].values.reshape(-1, 1)

    # weighted GMM via sample replication proxy: resample proportional to weight
    rng = np.random.default_rng(0)
    probs = weights / weights.sum()
    idx = rng.choice(len(seg), size=min(200000, 50 * len(seg)), p=probs)
    resampled = log2[idx]

    gmm = GaussianMixture(n_components=n_states, random_state=0).fit(resampled)
    means = gmm.means_.flatten()
    comp_weights = gmm.weights_

    # modal (most probable) log2 state = the sample's dominant copy number
    modal_log2 = means[np.argmax(comp_weights)]

    # anchor modal state to assumed_neutral_cn, then convert every segment's
    # log2 to an absolute copy number using that anchor:
    #   CN = assumed_neutral_cn * 2^(log2 - modal_log2)
    seg["cn_est"] = assumed_neutral_cn * (2 ** (seg["log2"] - modal_log2))
    weighted_ploidy = np.average(seg["cn_est"], weights=seg["length"])

    return {
        "modal_log2": modal_log2,
        "state_means": sorted(means.tolist()),
        "state_weights": comp_weights[np.argsort(means)].tolist(),
        "weighted_ploidy_estimate": weighted_ploidy,
    }, seg

ploidy_results = {}
seg_with_cn = {}
for s in SAMPLES:
    result, seg = estimate_ploidy(cns[s])
    ploidy_results[s] = result
    seg_with_cn[s] = seg
    print(s, "->", {k: v for k, v in result.items() if k != "state_means" and k != "state_weights"})

In [ ]:
fig, axes = plt.subplots(1, len(SAMPLES), figsize=(15, 4), sharey=True)
for ax, s in zip(axes, SAMPLES):
    ax.hist(cns[s]["log2"], weights=(cns[s]["end"] - cns[s]["start"]), bins=100, color=COLORS[s])
    ax.axvline(ploidy_results[s]["modal_log2"], color="black", linestyle="--", label="modal state")
    ax.set_title(f"{s}\nweighted ploidy ~ {ploidy_results[s]['weighted_ploidy_estimate']:.2f}")
    ax.set_xlabel("segment log2")
    ax.legend(fontsize=8)
axes[0].set_ylabel("genomic length (bp)")
plt.tight_layout()
plt.savefig(INPUT_DIR / "ploidy_estimate_histograms.png", dpi=200)
plt.show()

## 2b. Cross-check against ichorCNA

ichorCNA writes a `*.params.txt` per sample with its own purity/ploidy fit
(independent method, HMM over read depth). Compare its ploidy to the
histogram-based estimate above -- if they diverge a lot, treat both as
rough and lean on flow cytometry if available.

In [ ]:
ICHOR_DIR = INPUT_DIR / "ichorcna"

def parse_ichor_params(name):
    p = ICHOR_DIR / f"{name}.params.txt"
    if not p.exists():
        return None
    params = {}
    for line in p.read_text().splitlines():
        if ":" in line or "\t" in line:
            parts = line.replace(":", "\t").split("\t")
            if len(parts) >= 2:
                params[parts[0].strip()] = parts[1].strip()
    return params

for s in SAMPLES:
    params = parse_ichor_params(s)
    print(s, "ichorCNA params:", params if params else "not found (check ICHOR_DIR path)")

## 2c. Flow cytometry DNA content -- relative, high confidence; absolute, still assumed

No diploid standard was included, so the two peaks per sample (G1, G2/M)
still don't pin down absolute ploidy on their own -- G2/M is always ~2x G1
regardless of the sample's actual copy number, so without a co-run diploid
reference (normal PBMCs, RPE1) there's no external anchor for what "diploid"
looks like on this instrument/stain.

What upgrades this from a rough check to a trustworthy one: Parental,
Resistant, and Fused were all stained and acquired **in the same session**,
so batch drift isn't a live concern -- the measured G1 peak ratios between
samples are real, not staining/instrument artifacts.

Measured values (`scripts/analyze_flow_g1_peaks.py`, replicate-averaged from
Gaussian-refined G1 peaks; see `analysis/flow_cytometry/`):
- **Fused vs Parental: 1.62x.** Fused samples are literal cell-cell fusion
  hybrids, so a near-doubling of DNA content is exactly the expected
  signature (fusion produces a near-tetraploid cell; the shortfall from a
  clean 2.0x is consistent with the genome loss commonly seen as hybrid
  clones stabilize post-fusion). This is a positive control on the assay,
  not a surprise.
- **Resistant vs Parental: 0.83x.** A real, reproducible ~17% *decrease* in
  DNA content in the resistant subclone -- notable, since resistance
  mechanisms more often present as focal amplifications than genome-wide
  loss. Section 3b below checks whether the sequencing data shows this as a
  genome-wide shift (consistent with real ploidy loss) or something more
  localized once the CNVkit outputs are available.

`PARENTAL_PLOIDY_ASSUMPTION` below still anchors Parental at a stated,
flagged assumption (default near-diploid, 2N) purely to give the ratio-based
estimates absolute units -- the *ratios* above are trustworthy regardless of
whether that assumption holds; only the absolute numbers would shift
together if it's wrong.

In [ ]:
# Measured via scripts/analyze_flow_g1_peaks.py (replicate-averaged G1 peak
# channel, PE-A, from Gaussian refinement of the FlowJo "cells"-gated DNA
# histogram -- see analysis/flow_cytometry/g1_peak_per_replicate.csv for the
# per-replicate values and analysis/flow_cytometry/*_g1_peak.png for the
# diagnostic plots). Same instrument run/settings for all three, no diploid
# standard included.
FLOW_G1_PEAK = {
    "parental": 57853.5,   # replicates: 57486, 58221
    "resistant": 48306.3,  # replicates: 49009, 47604
    "fused": 93663.5,      # replicates: 93414, 93913
}

# No diploid standard was run, so absolute ploidy can't be read off these
# peaks directly. PARENTAL_PLOIDY_ASSUMPTION is an explicit, flagged
# ASSUMPTION (not a measurement) used only to give the ratio-based estimates
# below an absolute scale. Replace it if you re-run flow with a diploid
# control (normal PBMCs / RPE1) alongside the samples.
PARENTAL_PLOIDY_ASSUMPTION = 2.0

if any(v is None for v in FLOW_G1_PEAK.values()):
    print("Fill in FLOW_G1_PEAK with your measured G1 channel values to run this cell.")
else:
    flow_ratio_vs_parental = {s: FLOW_G1_PEAK[s] / FLOW_G1_PEAK["parental"] for s in SAMPLES}
    flow_anchored_ploidy = {s: PARENTAL_PLOIDY_ASSUMPTION * flow_ratio_vs_parental[s] for s in SAMPLES}

    print("G1 peak ratio vs Parental (relative, measured):", flow_ratio_vs_parental)
    print(f"Flow-anchored ploidy (Parental assumed {PARENTAL_PLOIDY_ASSUMPTION}N):", flow_anchored_ploidy)

    print("\nsequencing-only GMM estimate, for comparison:")
    for s in SAMPLES:
        print(f"  {s}: flow-anchored={flow_anchored_ploidy[s]:.2f}   seq-GMM={ploidy_results[s]['weighted_ploidy_estimate']:.2f}")

## 3. Relative CN: Resistant vs. Parental, Fused vs. Parental

Bin-level log2 difference against the isogenic Parental background -- this
is the direct readout of what changed during resistance / fusion
engineering, independent of the absolute-ploidy uncertainty above.

## 3b. Cross-check: does the flow-based ploidy ratio match the sequencing-based CN shift?

If Resistant/Fused underwent a genome-wide ploidy change relative to Parental
(e.g., whole-genome duplication), the median `delta_log2` across *all* bins in
section 3 should be centered near `log2(flow_ratio_vs_parental)`, not near 0.
If the flow ratio suggests a ploidy shift but the genome-wide median
delta_log2 is ~0, that's a real discrepancy worth resolving (subclonal
heterogeneity, a skewed cell-cycle mix in the flow sample, or the
near-diploid assumption above being wrong) rather than something to average
away.

In [ ]:
if any(v is None for v in FLOW_G1_PEAK.values()):
    print("FLOW_G1_PEAK not filled in yet -- skipping cross-check.")
else:
    for name, diff_df in [("resistant", diff_resistant), ("fused", diff_fused)]:
        observed_median_delta = diff_df["delta_log2"].median()
        expected_delta_from_flow = np.log2(flow_ratio_vs_parental[name])
        print(f"{name}: observed genome-wide median delta_log2={observed_median_delta:.3f}, "
              f"expected from flow ratio={expected_delta_from_flow:.3f}")

In [ ]:
def diff_vs_parental(other_name, bin_tol=1):
    p = cnr["parental"][["chromosome", "start", "end", "log2"]].rename(columns={"log2": "log2_parental"})
    o = cnr[other_name][["chromosome", "start", "end", "log2"]].rename(columns={"log2": f"log2_{other_name}"})
    merged = pd.merge(p, o, on=["chromosome", "start", "end"], how="inner")
    merged["delta_log2"] = merged[f"log2_{other_name}"] - merged["log2_parental"]
    return merged

diff_resistant = diff_vs_parental("resistant")
diff_fused = diff_vs_parental("fused")

# flag bins with a substantial acquired shift (>= 1 copy-equivalent change, |delta| > 0.4 log2 as a starting threshold)
THRESH = 0.4
acquired_resistant = diff_resistant[diff_resistant["delta_log2"].abs() > THRESH]
acquired_fused = diff_fused[diff_fused["delta_log2"].abs() > THRESH]

print(f"Resistant vs Parental: {len(acquired_resistant)} / {len(diff_resistant)} bins beyond |delta_log2|>{THRESH}")
print(f"Fused vs Parental: {len(acquired_fused)} / {len(diff_fused)} bins beyond |delta_log2|>{THRESH}")

acquired_resistant.to_csv(INPUT_DIR / "acquired_bins_resistant_vs_parental.csv", index=False)
acquired_fused.to_csv(INPUT_DIR / "acquired_bins_fused_vs_parental.csv", index=False)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

for ax, (diff_df, label) in zip(axes, [(diff_resistant, "Resistant - Parental"), (diff_fused, "Fused - Parental")]):
    diff_df, _ = add_genome_coord(diff_df, chrom_sizes)
    colors = np.where(diff_df["delta_log2"].abs() > THRESH, "crimson", "lightgrey")
    ax.scatter(diff_df["genome_start"], diff_df["delta_log2"], s=2, c=colors)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axhline(THRESH, color="crimson", linestyle="--", linewidth=0.5)
    ax.axhline(-THRESH, color="crimson", linestyle="--", linewidth=0.5)
    ax.set_ylabel(label)
    ax.set_ylim(-2, 2)

axes[-1].set_xticks([offsets[c] for c in CHROM_ORDER])
axes[-1].set_xticklabels(CHROM_ORDER, fontsize=8)
axes[-1].set_xlabel("Chromosome")
plt.tight_layout()
plt.savefig(INPUT_DIR / "acquired_cnv_vs_parental.png", dpi=200)
plt.show()

In [ ]:
import pysam

def load_baf(name):
    vcf = pysam.VariantFile(str(BAF_DIR / f"{name}.vcf.gz"))
    rows = []
    for rec in vcf:
        if len(rec.alleles) != 2:
            continue
        for sample in rec.samples.values():
            ad = sample.get("AD")
            if ad is None or len(ad) != 2 or sum(ad) == 0:
                continue
            baf = ad[1] / sum(ad)
            rows.append((str(rec.chrom).replace("chr", ""), rec.pos, baf, sum(ad)))
    return pd.DataFrame(rows, columns=["chromosome", "pos", "baf", "depth"])

baf = {}
for s in SAMPLES:
    try:
        baf[s] = load_baf(s)
        baf[s] = baf[s][baf[s]["depth"] >= 10]  # minimum depth filter
        print(s, "n_sites:", len(baf[s]))
    except FileNotFoundError:
        print(s, "BAF vcf not found -- run the pipeline's step 4 first")

In [ ]:
fig, axes = plt.subplots(len(SAMPLES), 1, figsize=(16, 3 * len(SAMPLES)), sharex=True)
for ax, s in zip(axes, SAMPLES):
    if s not in baf:
        continue
    df = baf[s].copy()
    df["chromosome"] = pd.Categorical(df["chromosome"], categories=CHROM_ORDER, ordered=True)
    df = df[df["chromosome"].notna()]
    df, _ = add_genome_coord(df.rename(columns={"pos": "start"}), chrom_sizes)
    ax.scatter(df["genome_start"], df["baf"], s=1, alpha=0.2, color=COLORS[s])
    ax.axhline(0.5, color="black", linewidth=0.5, linestyle="--")
    ax.set_ylim(0, 1)
    ax.set_ylabel(f"{s}\nBAF")

axes[-1].set_xticks([offsets[c] for c in CHROM_ORDER])
axes[-1].set_xticklabels(CHROM_ORDER, fontsize=8)
axes[-1].set_xlabel("Chromosome")
plt.tight_layout()
plt.savefig(INPUT_DIR / "baf_all_samples.png", dpi=200)
plt.show()

## 5. Summary table

In [ ]:
summary = pd.DataFrame({
    s: {
        "weighted_ploidy_estimate_seqGMM": ploidy_results[s]["weighted_ploidy_estimate"],
        "modal_log2": ploidy_results[s]["modal_log2"],
        "n_segments": len(cns[s]),
    }
    for s in SAMPLES
}).T

summary.loc["resistant", "n_acquired_bins_vs_parental"] = len(acquired_resistant)
summary.loc["fused", "n_acquired_bins_vs_parental"] = len(acquired_fused)

if not any(v is None for v in FLOW_G1_PEAK.values()):
    for s in SAMPLES:
        summary.loc[s, "flow_g1_peak"] = FLOW_G1_PEAK[s]
        summary.loc[s, "flow_anchored_ploidy"] = flow_anchored_ploidy[s]

summary.to_csv(INPUT_DIR / "cnv_ploidy_summary.csv")
summary